# Debugging Lab: Find the Bug!

This notebook contains 10 pre-broken code snippets with common optimizer bugs. Your task is to identify and fix each bug.

**Instructions:**
1. Read each buggy code snippet
2. Try to identify the bug before looking at hints
3. Expand the hints if you're stuck
4. Check your answer against the solution

**Common Bug Categories:**
- Wrong learning rate scale
- Gradient explosion
- Shape mismatches
- Missing zero_grad()
- Wrong loss function

---

In [ ]:
# Setup
import numpy as np
import sys
sys.path.insert(0, '..')

from src.optimizers import (
    SGD, Adam, AdamW, RMSprop, Adagrad,
    clip_grad_norm_, clip_grad_value_,
    compute_grad_norm, check_gradients
)

np.random.seed(42)
print("Setup complete!")

---

## Bug 1: The Learning Rate That's Too High

The following code tries to train a simple linear model, but the loss explodes instead of decreasing.

In [ ]:
# BUGGY CODE - Bug 1

def train_linear_model_buggy_1():
    # Generate simple data
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 1)
    
    # Initialize weights
    W = np.random.randn(10, 1)
    
    # BUG IS HERE - Can you spot it?
    optimizer = SGD([W], lr=10.0)  
    
    losses = []
    for epoch in range(100):
        # Forward pass
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        # Backward pass
        grad_W = (2 / len(X)) * X.T @ (y_pred - y)
        
        # Update
        optimizer.step([grad_W])
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
    
    return losses

# Run the buggy code
losses = train_linear_model_buggy_1()
print(f"\nFinal loss: {losses[-1]:.4f}")
print("Loss is exploding!" if losses[-1] > losses[0] else "Loss is decreasing")

<details>
<summary><b>Click for Hint 1</b></summary>

Look at the learning rate value. What's a typical learning rate for SGD?
</details>

<details>
<summary><b>Click for Hint 2</b></summary>

A learning rate of 10.0 is way too high for most problems. Try values between 0.001 and 0.1.
</details>

<details>
<summary><b>Click for Solution</b></summary>

**Bug:** The learning rate `lr=10.0` is too high, causing gradient updates to overshoot and the loss to explode.

**Fix:** Use a smaller learning rate like `lr=0.01` or `lr=0.1`.

```python
optimizer = SGD([W], lr=0.01)  # Fixed!
```

**Why this happens:** When the learning rate is too high, each update step moves the parameters too far, potentially jumping over the minimum and causing the loss to increase. This can cascade into exponential growth of the loss.
</details>

In [ ]:
# FIXED CODE - Bug 1

def train_linear_model_fixed_1():
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 1)
    W = np.random.randn(10, 1)
    
    # FIXED: Use appropriate learning rate
    optimizer = SGD([W], lr=0.01)
    
    losses = []
    for epoch in range(100):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        grad_W = (2 / len(X)) * X.T @ (y_pred - y)
        optimizer.step([grad_W])
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
    return losses

print("Fixed version:")
losses_fixed = train_linear_model_fixed_1()
print(f"\nFinal loss: {losses_fixed[-1]:.4f}")

---

## Bug 2: Exploding Gradients in RNN-like Computation

This code simulates an RNN-like computation where gradients explode through time.

In [ ]:
# BUGGY CODE - Bug 2

def train_rnn_buggy():
    # Simulate RNN weights
    W_hh = np.random.randn(50, 50) * 1.5  # Hidden-to-hidden weights
    W_xh = np.random.randn(10, 50) * 0.1  # Input-to-hidden weights
    
    optimizer = Adam([W_hh, W_xh], lr=0.001)
    
    for epoch in range(10):
        # Simulate forward pass through time
        h = np.zeros(50)
        x_seq = np.random.randn(20, 10)  # Sequence of 20 timesteps
        
        # Simulate backward pass (gradient computation)
        # In reality, this would use backprop through time
        grad_W_hh = np.zeros_like(W_hh)
        grad_W_xh = np.zeros_like(W_xh)
        
        # Simulate accumulating gradients through time
        multiplier = 1.0
        for t in range(20):
            # Gradients compound through time
            multiplier *= np.linalg.norm(W_hh)  # This causes explosion!
            grad_W_hh += np.random.randn(50, 50) * multiplier
            grad_W_xh += np.random.randn(10, 50) * multiplier
        
        # BUG: No gradient clipping!
        grads = [grad_W_hh, grad_W_xh]
        
        grad_norm = compute_grad_norm(grads)
        print(f"Epoch {epoch}: Gradient norm = {grad_norm:.2e}")
        
        if np.isnan(grad_norm) or np.isinf(grad_norm):
            print("Gradients exploded!")
            break
            
        optimizer.step(grads)

print("Running buggy RNN training...")
train_rnn_buggy()

<details>
<summary><b>Click for Hint 1</b></summary>

RNNs are notorious for exploding gradients. What technique is commonly used to prevent this?
</details>

<details>
<summary><b>Click for Hint 2</b></summary>

Use `clip_grad_norm_()` to limit the magnitude of gradients before the optimizer step.
</details>

<details>
<summary><b>Click for Solution</b></summary>

**Bug:** No gradient clipping is applied, allowing gradients to grow exponentially through time.

**Fix:** Add gradient clipping before the optimizer step:

```python
grads = [grad_W_hh, grad_W_xh]
clip_grad_norm_(grads, max_norm=1.0)  # Add this line!
optimizer.step(grads)
```

**Why this happens:** In RNNs, gradients are multiplied by the weight matrix at each timestep during backpropagation. If the spectral norm of the weight matrix is > 1, gradients grow exponentially. Gradient clipping limits this growth.
</details>

In [ ]:
# FIXED CODE - Bug 2

def train_rnn_fixed():
    W_hh = np.random.randn(50, 50) * 1.5
    W_xh = np.random.randn(10, 50) * 0.1
    
    optimizer = Adam([W_hh, W_xh], lr=0.001)
    
    for epoch in range(10):
        h = np.zeros(50)
        x_seq = np.random.randn(20, 10)
        
        grad_W_hh = np.zeros_like(W_hh)
        grad_W_xh = np.zeros_like(W_xh)
        
        multiplier = 1.0
        for t in range(20):
            multiplier *= np.linalg.norm(W_hh)
            grad_W_hh += np.random.randn(50, 50) * multiplier
            grad_W_xh += np.random.randn(10, 50) * multiplier
        
        grads = [grad_W_hh, grad_W_xh]
        
        # FIXED: Add gradient clipping!
        clip_grad_norm_(grads, max_norm=1.0)
        
        grad_norm = compute_grad_norm(grads)
        print(f"Epoch {epoch}: Gradient norm = {grad_norm:.4f}")
        
        optimizer.step(grads)

print("Running fixed RNN training...")
train_rnn_fixed()

---

## Bug 3: Shape Mismatch in Gradient

The following code has a subtle shape mismatch between parameters and gradients.

In [ ]:
# BUGGY CODE - Bug 3

def train_mlp_buggy():
    # Two-layer MLP
    W1 = np.random.randn(10, 20)  # Input -> Hidden
    W2 = np.random.randn(20, 5)   # Hidden -> Output
    
    optimizer = Adam([W1, W2], lr=0.001)
    
    X = np.random.randn(32, 10)  # Batch of 32 samples
    y = np.random.randn(32, 5)
    
    # Forward pass
    h = np.maximum(0, X @ W1)  # ReLU activation
    y_pred = h @ W2
    
    # Backward pass - BUG IS HERE!
    grad_output = (y_pred - y)  # Shape: (32, 5)
    
    # Gradient for W2
    grad_W2 = h.T @ grad_output  # Shape: (20, 5) - Correct!
    
    # Gradient for W1 - BUG!
    grad_h = grad_output @ W2.T  # Shape: (32, 20)
    grad_h[h <= 0] = 0  # ReLU backward
    grad_W1 = grad_h.T @ X  # BUG: Wrong order! Should be X.T @ grad_h
    
    print(f"W1 shape: {W1.shape}")
    print(f"grad_W1 shape: {grad_W1.shape}")
    print(f"Shapes match: {W1.shape == grad_W1.shape}")
    
    try:
        optimizer.step([grad_W1, grad_W2])
        print("Optimizer step succeeded")
    except Exception as e:
        print(f"Error: {e}")

train_mlp_buggy()

<details>
<summary><b>Click for Hint 1</b></summary>

Compare the shape of `W1` with the shape of `grad_W1`. Do they match?
</details>

<details>
<summary><b>Click for Hint 2</b></summary>

The gradient computation for W1 has the matrix multiplication in the wrong order. For a weight matrix W where output = input @ W, the gradient is input.T @ grad_output.
</details>

<details>
<summary><b>Click for Solution</b></summary>

**Bug:** `grad_W1 = grad_h.T @ X` produces wrong shape (20, 10) instead of (10, 20).

**Fix:** Change to `grad_W1 = X.T @ grad_h` which produces correct shape (10, 20).

```python
grad_W1 = X.T @ grad_h  # Fixed! Shape: (10, 20)
```

**Why this happens:** For a linear layer `y = X @ W`, the gradient w.r.t. W is `X.T @ grad_y`. The order matters because matrix multiplication is not commutative.
</details>

In [ ]:
# FIXED CODE - Bug 3

def train_mlp_fixed():
    W1 = np.random.randn(10, 20)
    W2 = np.random.randn(20, 5)
    
    optimizer = Adam([W1, W2], lr=0.001)
    
    X = np.random.randn(32, 10)
    y = np.random.randn(32, 5)
    
    h = np.maximum(0, X @ W1)
    y_pred = h @ W2
    
    grad_output = (y_pred - y)
    grad_W2 = h.T @ grad_output
    
    grad_h = grad_output @ W2.T
    grad_h[h <= 0] = 0
    
    # FIXED: Correct order for gradient computation
    grad_W1 = X.T @ grad_h  # Shape: (10, 20) - Correct!
    
    print(f"W1 shape: {W1.shape}")
    print(f"grad_W1 shape: {grad_W1.shape}")
    print(f"Shapes match: {W1.shape == grad_W1.shape}")
    
    optimizer.step([grad_W1, grad_W2])
    print("Optimizer step succeeded!")

train_mlp_fixed()

---

## Bug 4: Accumulating Gradients (Missing zero_grad)

This code accumulates gradients across iterations instead of resetting them.

In [ ]:
# BUGGY CODE - Bug 4

def train_with_accumulated_grads_buggy():
    W = np.random.randn(10, 5)
    optimizer = SGD([W], lr=0.01)
    
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 5)
    
    # BUG: Gradient is defined outside the loop and never reset!
    accumulated_grad = np.zeros_like(W)
    
    losses = []
    for epoch in range(50):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        # Compute gradient
        grad_W = (2 / len(X)) * X.T @ (y_pred - y)
        
        # BUG: Accumulating instead of replacing!
        accumulated_grad += grad_W
        
        optimizer.step([accumulated_grad])
        
        if epoch % 10 == 0:
            grad_norm = np.linalg.norm(accumulated_grad)
            print(f"Epoch {epoch}: Loss = {loss:.4f}, Grad norm = {grad_norm:.4f}")
    
    return losses

print("Buggy version (accumulating gradients):")
losses_buggy = train_with_accumulated_grads_buggy()

<details>
<summary><b>Click for Hint 1</b></summary>

What happens to `accumulated_grad` across iterations? Is it ever reset?
</details>

<details>
<summary><b>Click for Hint 2</b></summary>

In PyTorch, you call `optimizer.zero_grad()` before computing new gradients. Here, we're manually managing gradients, so we need to reset them ourselves.
</details>

<details>
<summary><b>Click for Solution</b></summary>

**Bug:** `accumulated_grad` is never reset, so gradients from all previous iterations accumulate.

**Fix:** Either:
1. Use the current gradient directly: `optimizer.step([grad_W])`
2. Or reset the accumulator: `accumulated_grad = np.zeros_like(W)` at the start of each iteration

```python
# Option 1: Use gradient directly
optimizer.step([grad_W])

# Option 2: Reset accumulator (if you need it for gradient accumulation)
accumulated_grad = np.zeros_like(W)  # Reset at start of each iteration
accumulated_grad += grad_W
optimizer.step([accumulated_grad])
```

**Why this happens:** In deep learning frameworks, gradients accumulate by default to support gradient accumulation over multiple mini-batches. You must explicitly zero them before computing new gradients.
</details>

In [ ]:
# FIXED CODE - Bug 4

def train_with_accumulated_grads_fixed():
    W = np.random.randn(10, 5)
    optimizer = SGD([W], lr=0.01)
    
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 5)
    
    losses = []
    for epoch in range(50):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        grad_W = (2 / len(X)) * X.T @ (y_pred - y)
        
        # FIXED: Use gradient directly without accumulation
        optimizer.step([grad_W])
        
        if epoch % 10 == 0:
            grad_norm = np.linalg.norm(grad_W)
            print(f"Epoch {epoch}: Loss = {loss:.4f}, Grad norm = {grad_norm:.4f}")
    
    return losses

print("Fixed version:")
losses_fixed = train_with_accumulated_grads_fixed()

---

## Bug 5: Wrong Loss Function for Classification

This code uses MSE loss for a classification problem.

In [ ]:
# BUGGY CODE - Bug 5

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def train_classifier_buggy():
    # Classification problem: 3 classes
    W = np.random.randn(10, 3) * 0.01
    optimizer = Adam([W], lr=0.01)
    
    # Generate classification data
    X = np.random.randn(100, 10)
    y_true = np.random.randint(0, 3, 100)  # Class labels 0, 1, 2
    
    # One-hot encode
    y_onehot = np.zeros((100, 3))
    y_onehot[np.arange(100), y_true] = 1
    
    losses = []
    for epoch in range(100):
        # Forward pass
        logits = X @ W
        probs = softmax(logits)
        
        # BUG: Using MSE loss for classification!
        loss = np.mean((probs - y_onehot) ** 2)
        losses.append(loss)
        
        # Gradient of MSE (not ideal for classification)
        grad_probs = 2 * (probs - y_onehot) / len(X)
        grad_W = X.T @ grad_probs
        
        optimizer.step([grad_W])
        
        if epoch % 20 == 0:
            preds = np.argmax(probs, axis=1)
            accuracy = np.mean(preds == y_true)
            print(f"Epoch {epoch}: Loss = {loss:.4f}, Accuracy = {accuracy:.2%}")
    
    return losses

print("Buggy version (MSE loss for classification):")
losses_buggy = train_classifier_buggy()

<details>
<summary><b>Click for Hint 1</b></summary>

What loss function is typically used for multi-class classification? Think about what loss pairs well with softmax.
</details>

<details>
<summary><b>Click for Hint 2</b></summary>

Cross-entropy loss is the standard choice for classification. It has nice properties when combined with softmax (the gradient simplifies to `probs - y_onehot`).
</details>

<details>
<summary><b>Click for Solution</b></summary>

**Bug:** Using MSE loss for classification. MSE doesn't account for the probabilistic nature of the output and provides weak gradients when predictions are confident but wrong.

**Fix:** Use cross-entropy loss:

```python
# Cross-entropy loss
loss = -np.mean(np.sum(y_onehot * np.log(probs + 1e-8), axis=1))

# Gradient of cross-entropy with softmax (simplifies nicely!)
grad_logits = (probs - y_onehot) / len(X)
grad_W = X.T @ grad_logits
```

**Why this matters:** Cross-entropy loss:
- Penalizes confident wrong predictions heavily
- Has a simple gradient when combined with softmax
- Is derived from maximum likelihood estimation for categorical distributions
</details>

In [ ]:
# FIXED CODE - Bug 5

def train_classifier_fixed():
    W = np.random.randn(10, 3) * 0.01
    optimizer = Adam([W], lr=0.01)
    
    X = np.random.randn(100, 10)
    y_true = np.random.randint(0, 3, 100)
    y_onehot = np.zeros((100, 3))
    y_onehot[np.arange(100), y_true] = 1
    
    losses = []
    for epoch in range(100):
        logits = X @ W
        probs = softmax(logits)
        
        # FIXED: Use cross-entropy loss
        loss = -np.mean(np.sum(y_onehot * np.log(probs + 1e-8), axis=1))
        losses.append(loss)
        
        # FIXED: Gradient of cross-entropy + softmax
        grad_logits = (probs - y_onehot) / len(X)
        grad_W = X.T @ grad_logits
        
        optimizer.step([grad_W])
        
        if epoch % 20 == 0:
            preds = np.argmax(probs, axis=1)
            accuracy = np.mean(preds == y_true)
            print(f"Epoch {epoch}: Loss = {loss:.4f}, Accuracy = {accuracy:.2%}")
    
    return losses

print("Fixed version (cross-entropy loss):")
losses_fixed = train_classifier_fixed()

---

## Bug 6: Adam with Wrong Beta Values

This code uses incorrect beta values for Adam, causing slow convergence.

In [ ]:
# BUGGY CODE - Bug 6

def train_with_wrong_betas():
    W = np.random.randn(20, 10)
    
    # BUG: beta1 and beta2 are swapped/wrong!
    optimizer = Adam([W], lr=0.001, betas=(0.999, 0.9))
    
    X = np.random.randn(100, 20)
    y = np.random.randn(100, 10)
    
    losses = []
    for epoch in range(100):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        grad_W = (2 / len(X)) * X.T @ (y_pred - y)
        optimizer.step([grad_W])
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
    
    return losses

print("Buggy version (wrong betas):")
losses_buggy = train_with_wrong_betas()

<details>
<summary><b>Click for Hint 1</b></summary>

What are the standard values for beta1 and beta2 in Adam? Which one controls the first moment (momentum) and which controls the second moment?
</details>

<details>
<summary><b>Click for Hint 2</b></summary>

Standard Adam uses beta1=0.9 (first moment/momentum) and beta2=0.999 (second moment/adaptive learning rate). The values in the buggy code are swapped!
</details>

<details>
<summary><b>Click for Solution</b></summary>

**Bug:** Beta values are swapped: `betas=(0.999, 0.9)` instead of `betas=(0.9, 0.999)`.

**Fix:**
```python
optimizer = Adam([W], lr=0.001, betas=(0.9, 0.999))  # Standard values
```

**Why this matters:**
- beta1 controls momentum (first moment). 0.9 means gradients are averaged over ~10 steps.
- beta2 controls adaptive learning rate (second moment). 0.999 means squared gradients are averaged over ~1000 steps for stability.
- Swapping them makes momentum too slow and the adaptive rate too noisy.
</details>

In [ ]:
# FIXED CODE - Bug 6

def train_with_correct_betas():
    W = np.random.randn(20, 10)
    
    # FIXED: Correct beta values
    optimizer = Adam([W], lr=0.001, betas=(0.9, 0.999))
    
    X = np.random.randn(100, 20)
    y = np.random.randn(100, 10)
    
    losses = []
    for epoch in range(100):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        grad_W = (2 / len(X)) * X.T @ (y_pred - y)
        optimizer.step([grad_W])
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
    
    return losses

print("Fixed version (correct betas):")
losses_fixed = train_with_correct_betas()

---

## Bug 7: Vanishing Gradients from Bad Initialization

This code has weights initialized too large, causing activation saturation and vanishing gradients.

In [ ]:
# BUGGY CODE - Bug 7

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def train_deep_net_buggy():
    # Deep network with sigmoid activations
    # BUG: Weights initialized too large!
    W1 = np.random.randn(10, 100) * 5.0  # Too large!
    W2 = np.random.randn(100, 100) * 5.0
    W3 = np.random.randn(100, 10) * 5.0
    
    optimizer = Adam([W1, W2, W3], lr=0.001)
    
    X = np.random.randn(32, 10)
    y = np.random.randn(32, 10)
    
    for epoch in range(50):
        # Forward pass
        h1 = sigmoid(X @ W1)
        h2 = sigmoid(h1 @ W2)
        y_pred = h2 @ W3
        
        loss = np.mean((y_pred - y) ** 2)
        
        # Backward pass
        grad_y = 2 * (y_pred - y) / len(X)
        grad_W3 = h2.T @ grad_y
        
        grad_h2 = grad_y @ W3.T
        grad_h2 *= h2 * (1 - h2)  # sigmoid derivative
        grad_W2 = h1.T @ grad_h2
        
        grad_h1 = grad_h2 @ W2.T
        grad_h1 *= h1 * (1 - h1)
        grad_W1 = X.T @ grad_h1
        
        grads = [grad_W1, grad_W2, grad_W3]
        
        if epoch % 10 == 0:
            grad_norms = [np.linalg.norm(g) for g in grads]
            print(f"Epoch {epoch}: Loss = {loss:.4f}, Grad norms = {grad_norms}")
            print(f"  h1 mean: {h1.mean():.4f}, h2 mean: {h2.mean():.4f}")
        
        optimizer.step(grads)

print("Buggy version (bad initialization):")
train_deep_net_buggy()

<details>
<summary><b>Click for Hint 1</b></summary>

Look at the activation values (h1 mean, h2 mean). What happens when sigmoid inputs are very large or very small?
</details>

<details>
<summary><b>Click for Hint 2</b></summary>

Sigmoid saturates at 0 and 1 for large inputs. When saturated, the derivative `sigmoid(x) * (1 - sigmoid(x))` is nearly 0, causing vanishing gradients. Use smaller initial weights or Xavier/He initialization.
</details>

<details>
<summary><b>Click for Solution</b></summary>

**Bug:** Weights initialized with scale 5.0, which is way too large. This causes sigmoid activations to saturate (close to 0 or 1), where the gradient is nearly zero.

**Fix:** Use proper initialization like Xavier/Glorot:

```python
# Xavier initialization for sigmoid/tanh
W1 = np.random.randn(10, 100) * np.sqrt(2.0 / (10 + 100))
W2 = np.random.randn(100, 100) * np.sqrt(2.0 / (100 + 100))
W3 = np.random.randn(100, 10) * np.sqrt(2.0 / (100 + 10))
```

**Why this happens:** Large weights -> large pre-activation values -> sigmoid saturation -> gradient = sigmoid * (1 - sigmoid) approaches 0 -> vanishing gradients.
</details>

In [ ]:
# FIXED CODE - Bug 7

def train_deep_net_fixed():
    # FIXED: Xavier initialization
    W1 = np.random.randn(10, 100) * np.sqrt(2.0 / (10 + 100))
    W2 = np.random.randn(100, 100) * np.sqrt(2.0 / (100 + 100))
    W3 = np.random.randn(100, 10) * np.sqrt(2.0 / (100 + 10))
    
    optimizer = Adam([W1, W2, W3], lr=0.001)
    
    X = np.random.randn(32, 10)
    y = np.random.randn(32, 10)
    
    for epoch in range(50):
        h1 = sigmoid(X @ W1)
        h2 = sigmoid(h1 @ W2)
        y_pred = h2 @ W3
        
        loss = np.mean((y_pred - y) ** 2)
        
        grad_y = 2 * (y_pred - y) / len(X)
        grad_W3 = h2.T @ grad_y
        
        grad_h2 = grad_y @ W3.T
        grad_h2 *= h2 * (1 - h2)
        grad_W2 = h1.T @ grad_h2
        
        grad_h1 = grad_h2 @ W2.T
        grad_h1 *= h1 * (1 - h1)
        grad_W1 = X.T @ grad_h1
        
        grads = [grad_W1, grad_W2, grad_W3]
        
        if epoch % 10 == 0:
            grad_norms = [np.linalg.norm(g) for g in grads]
            print(f"Epoch {epoch}: Loss = {loss:.4f}, Grad norms = {[f'{n:.4f}' for n in grad_norms]}")
            print(f"  h1 mean: {h1.mean():.4f}, h2 mean: {h2.mean():.4f}")
        
        optimizer.step(grads)

print("Fixed version (Xavier initialization):")
train_deep_net_fixed()

---

## Bug 8: Forgetting to Update Parameters

This code computes gradients but never actually updates the parameters.

In [ ]:
# BUGGY CODE - Bug 8

def train_without_update_buggy():
    W = np.random.randn(10, 5)
    optimizer = SGD([W], lr=0.01)
    
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 5)
    
    losses = []
    for epoch in range(50):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        grad_W = (2 / len(X)) * X.T @ (y_pred - y)
        
        # BUG: Creating a NEW array instead of modifying W!
        W = W - 0.01 * grad_W  # This creates a new array, doesn't modify original!
        
        # The optimizer still has reference to the OLD W
        # optimizer.step([grad_W])  # This would work, but it's commented out
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
    
    # Check if optimizer's parameters changed
    print(f"\nOptimizer's W[0,0]: {optimizer.param_groups[0]['params'][0][0,0]:.4f}")
    print(f"Local W[0,0]: {W[0,0]:.4f}")
    print("They're different! Optimizer never got updates.")
    
    return losses

print("Buggy version (not updating through optimizer):")
losses_buggy = train_without_update_buggy()

<details>
<summary><b>Click for Hint 1</b></summary>

Look at how W is being updated. Is it using the optimizer, or doing something else?
</details>

<details>
<summary><b>Click for Hint 2</b></summary>

`W = W - 0.01 * grad_W` creates a NEW array and rebinds the variable W. The optimizer still holds a reference to the ORIGINAL array. Use `optimizer.step()` or modify in-place with `W -= 0.01 * grad_W`.
</details>

<details>
<summary><b>Click for Solution</b></summary>

**Bug:** `W = W - 0.01 * grad_W` creates a new array instead of modifying the original. The optimizer holds a reference to the original array, which never gets updated.

**Fix:** Use the optimizer to update parameters:

```python
optimizer.step([grad_W])  # Use optimizer
```

Or use in-place update:
```python
W -= 0.01 * grad_W  # Modifies W in-place
```

**Why this happens:** In Python, `a = a - b` creates a new object and rebinds the variable. `a -= b` modifies in-place. When using optimizers, always use `optimizer.step()` to ensure the correct arrays are modified.
</details>

In [ ]:
# FIXED CODE - Bug 8

def train_without_update_fixed():
    W = np.random.randn(10, 5)
    optimizer = SGD([W], lr=0.01)
    
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 5)
    
    losses = []
    for epoch in range(50):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        grad_W = (2 / len(X)) * X.T @ (y_pred - y)
        
        # FIXED: Use optimizer to update parameters
        optimizer.step([grad_W])
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
    
    print(f"\nOptimizer's W[0,0]: {optimizer.param_groups[0]['params'][0][0,0]:.4f}")
    print(f"Local W[0,0]: {W[0,0]:.4f}")
    print("They're the same! Optimizer updates work correctly.")
    
    return losses

print("Fixed version:")
losses_fixed = train_without_update_fixed()

---

## Bug 9: NaN from Log of Zero

This code produces NaN values due to taking log of zero.

In [ ]:
# BUGGY CODE - Bug 9

def train_with_nan_buggy():
    W = np.random.randn(10, 3) * 0.01
    optimizer = Adam([W], lr=0.1)  # High LR to trigger the bug faster
    
    X = np.random.randn(100, 10)
    y_true = np.random.randint(0, 3, 100)
    y_onehot = np.zeros((100, 3))
    y_onehot[np.arange(100), y_true] = 1
    
    for epoch in range(20):
        logits = X @ W
        probs = softmax(logits)
        
        # BUG: No epsilon in log! Can get log(0) = -inf
        loss = -np.mean(np.sum(y_onehot * np.log(probs), axis=1))
        
        grad_logits = (probs - y_onehot) / len(X)
        grad_W = X.T @ grad_logits
        
        print(f"Epoch {epoch}: Loss = {loss:.4f}, Has NaN: {np.isnan(loss)}")
        
        if np.isnan(loss):
            print("NaN detected! Training failed.")
            print(f"Min prob: {probs.min():.2e}")
            break
        
        optimizer.step([grad_W])

print("Buggy version (log without epsilon):")
train_with_nan_buggy()

<details>
<summary><b>Click for Hint 1</b></summary>

What happens when you compute `log(0)`? What could cause a probability to be exactly 0?
</details>

<details>
<summary><b>Click for Hint 2</b></summary>

Softmax can produce values very close to 0 due to numerical precision. Add a small epsilon inside the log: `log(probs + 1e-8)`.
</details>

<details>
<summary><b>Click for Solution</b></summary>

**Bug:** `np.log(probs)` can produce `-inf` when probs contains zeros, leading to NaN in subsequent calculations.

**Fix:** Add epsilon for numerical stability:

```python
loss = -np.mean(np.sum(y_onehot * np.log(probs + 1e-8), axis=1))
```

**Why this happens:** Softmax can produce extremely small values (effectively 0 in float32) when logits have large differences. Taking log(0) = -inf, and any operation with -inf can produce NaN.
</details>

In [ ]:
# FIXED CODE - Bug 9

def train_with_nan_fixed():
    W = np.random.randn(10, 3) * 0.01
    optimizer = Adam([W], lr=0.1)
    
    X = np.random.randn(100, 10)
    y_true = np.random.randint(0, 3, 100)
    y_onehot = np.zeros((100, 3))
    y_onehot[np.arange(100), y_true] = 1
    
    for epoch in range(20):
        logits = X @ W
        probs = softmax(logits)
        
        # FIXED: Add epsilon for numerical stability
        loss = -np.mean(np.sum(y_onehot * np.log(probs + 1e-8), axis=1))
        
        grad_logits = (probs - y_onehot) / len(X)
        grad_W = X.T @ grad_logits
        
        print(f"Epoch {epoch}: Loss = {loss:.4f}, Has NaN: {np.isnan(loss)}")
        
        optimizer.step([grad_W])

print("Fixed version (log with epsilon):")
train_with_nan_fixed()

---

## Bug 10: Wrong Gradient Sign

This code has the gradient sign wrong, causing the loss to increase instead of decrease.

In [ ]:
# BUGGY CODE - Bug 10

def train_with_wrong_sign_buggy():
    W = np.random.randn(10, 5) * 0.1
    optimizer = SGD([W], lr=0.01)
    
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 5)
    
    losses = []
    for epoch in range(50):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        # BUG: Wrong sign! Should be (y_pred - y), not (y - y_pred)
        grad_W = (2 / len(X)) * X.T @ (y - y_pred)  # WRONG SIGN!
        
        optimizer.step([grad_W])
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
    
    print(f"\nLoss is {'increasing' if losses[-1] > losses[0] else 'decreasing'}!")
    return losses

print("Buggy version (wrong gradient sign):")
losses_buggy = train_with_wrong_sign_buggy()

<details>
<summary><b>Click for Hint 1</b></summary>

The gradient should point in the direction of steepest INCREASE of the loss. We subtract it to DECREASE the loss. Check if the gradient formula is correct.
</details>

<details>
<summary><b>Click for Hint 2</b></summary>

For MSE loss = (1/n) * sum((y_pred - y)^2), the gradient w.r.t. y_pred is (2/n) * (y_pred - y), not (y - y_pred).
</details>

<details>
<summary><b>Click for Solution</b></summary>

**Bug:** The gradient is computed as `(y - y_pred)` but it should be `(y_pred - y)`.

**Fix:**
```python
grad_W = (2 / len(X)) * X.T @ (y_pred - y)  # Correct sign
```

**Why this happens:** The gradient of MSE w.r.t. predictions is `2 * (y_pred - y)`. Using the wrong sign means we're doing gradient ASCENT instead of gradient DESCENT, maximizing the loss instead of minimizing it.
</details>

In [ ]:
# FIXED CODE - Bug 10

def train_with_wrong_sign_fixed():
    W = np.random.randn(10, 5) * 0.1
    optimizer = SGD([W], lr=0.01)
    
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 5)
    
    losses = []
    for epoch in range(50):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        # FIXED: Correct gradient sign
        grad_W = (2 / len(X)) * X.T @ (y_pred - y)
        
        optimizer.step([grad_W])
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
    
    print(f"\nLoss is {'increasing' if losses[-1] > losses[0] else 'decreasing'}!")
    return losses

print("Fixed version (correct gradient sign):")
losses_fixed = train_with_wrong_sign_fixed()

---

## Summary: Common Optimizer Bugs

| Bug | Symptom | Fix |
|-----|---------|-----|
| Learning rate too high | Loss explodes | Use smaller LR (1e-3 to 1e-1) |
| Missing gradient clipping | Gradient norms explode (RNNs) | Use `clip_grad_norm_()` |
| Shape mismatch | Runtime error or silent bugs | Check gradient shapes match params |
| Accumulating gradients | Loss doesn't decrease properly | Reset gradients each iteration |
| Wrong loss function | Poor convergence on classification | Use cross-entropy for classification |
| Wrong Adam betas | Slow or unstable convergence | Use (0.9, 0.999) as default |
| Bad initialization | Vanishing/exploding gradients | Use Xavier or He initialization |
| Not using optimizer | Parameters don't update | Use `optimizer.step()` |
| Log of zero | NaN values | Add epsilon: `log(x + 1e-8)` |
| Wrong gradient sign | Loss increases | Double-check gradient derivation |

---
*Generated for "Optimization for Machine Learning" book*